# 其他13谱系CPU L2 + 元数据CSV
请把下面代码复制到仍有巨噬细胞数据的原Notebook中。无需重启kernel。先运行1，再运行2；3仅用于看进度。

In [ ]:
# Cell 1: 只读导出 study/sample/临床信息；不清空kernel，不读取表达矩阵。
from pathlib import Path
import importlib.util

OTHER_L2_ROOT = Path(
    "/home/anzhen_vhd/scRNAseq/codex/"
    "panvascular_atlas_rebuild_v2_20260907/"
    "06_reanalysis_takeover_20260908"
)
OTHER_L2_CODE = OTHER_L2_ROOT / "Other_L2_CPU_overnight_code"
OTHER_L2_SOURCE = OTHER_L2_ROOT / "panvascular_CPU_theta2_level1_frozen_v1.h5ad"

_other_meta_spec = importlib.util.spec_from_file_location(
    "other_l2_metadata_export", OTHER_L2_CODE / "sample_metadata_export.py"
)
other_l2_metadata = importlib.util.module_from_spec(_other_meta_spec)
_other_meta_spec.loader.exec_module(other_l2_metadata)

SAMPLE_METADATA_EXPORT = other_l2_metadata.export_sample_metadata(
    source=OTHER_L2_SOURCE
)
print("主CSV：", SAMPLE_METADATA_EXPORT / "sample_study_clinical_metadata.csv")


In [ ]:
# Cell 2: 启动其他13个谱系的独立CPU批处理，不改动巨噬细胞对象/结果。
from pathlib import Path
import importlib.util

OTHER_L2_ROOT = Path(
    "/home/anzhen_vhd/scRNAseq/codex/"
    "panvascular_atlas_rebuild_v2_20260907/"
    "06_reanalysis_takeover_20260908"
)
OTHER_L2_CODE = OTHER_L2_ROOT / "Other_L2_CPU_overnight_code"
_other_batch_spec = importlib.util.spec_from_file_location(
    "other_l2_batch_driver", OTHER_L2_CODE / "other_lineages_cpu_batch.py"
)
other_l2_batch = importlib.util.module_from_spec(_other_batch_spec)
_other_batch_spec.loader.exec_module(other_l2_batch)

OTHER_L2_RUN, OTHER_L2_PID = other_l2_batch.launch_other_lineages(
    source=OTHER_L2_ROOT / "panvascular_CPU_theta2_level1_frozen_v1.h5ad",
    max_workers=2,
    total_threads=40,
    resume_run=globals().get("OTHER_L2_RUN"),
    macrophage_run=globals().get("MACROPHAGE_L2_RUN"),
)
print("其他谱系输出：", OTHER_L2_RUN)
print("调度器日志：", OTHER_L2_RUN / "scheduler.log")
print("当前巨噬细胞变量未清除；无需保持本cell一直显示输出。")


In [ ]:
# Cell 3: 仅查看进度，可以明早再运行，也可随时重复，不会启动重复任务。
OTHER_L2_PROGRESS = other_l2_batch.batch_status(OTHER_L2_RUN)
display(OTHER_L2_PROGRESS)
